## CASSDA - Data Download

Contains code to download the following data
- Wildlife crossings from OSM
- Streets from OSM
- High-Resolution satellite images from discomap https://discomap.eea.europa.eu/wiki/ (Cloudless 2m/px satellite data) based on the download wildlife crossings from OSM

#### Deps

In [1]:
import os

import geopandas as gpd
import requests as r
from owslib.wms import WebMapService
from shapely.geometry import LineString, MultiPolygon, Point, Polygon

from detect_wildlife_crossings.modelling.helpers import polygon_to_yolo_segmentation
from detect_wildlife_crossings.osm.queries import (
    get_street_query,
    get_wildlife_crossing_query,
)
from detect_wildlife_crossings.wms.helpers import get_crop_bboxes

### Config

In [8]:
# OSM Download (shared settings)
# ------------------------------------------------------------
COUNTRIES_TO_PROCESS = ["CH"]
GEODATA_OUTPUT_DIR = "../data/geodata/"
OVERPASS_API_URL = "https://overpass-api.de/api/interpreter"

# OSM - Wildlife Crossings
BRIDGES_ONLY = True  # whether to only consider bridges or tunnels and other as well

# OSM - Streets
STREET_TYPE_DEPTH = (
    1  # how detailed the street network should be (see tag "highway" in OSM)
)

# WMS Download
# ------------------------------------------------------------
IMAGE_OUTPUT_DIR = "../data/satellite_images/"
PIXEL_RESOLUTION_M = 2  # meter per pixel
BUFFER_SIZE = 640  # Target image size in pixels (will be BUFFER_SIZE x BUFFER_SIZE)

In [13]:
# These typically do not need to be changed
WMS_URL = "https://image.discomap.eea.europa.eu/arcgis/services/GioLand/VHR_2021_LAEA/ImageServer/WMSServer/?request=GetCapabilities&service=WMS"
CRS = "EPSG:3035"  # ETRS89 / ETRS-LAEA
OVERPASS_TIMEOUT = 180  # timeout for overpass query in seconds

bridges_string = "_bridges_only" if BRIDGES_ONLY else ""
SOURCE_GPKGS = [
    f"../data/geodata/{country.lower()}_wildlife_crossings_osm{bridges_string}.gpkg"
    for country in COUNTRIES_TO_PROCESS
]

### Wildlife Crossing

#### Export Polygons from OSM

Gets the queries according to defined parameters, sends it co Overpass-API and 
on success stores the result in <geodata_output_dir>

In [16]:
for country in COUNTRIES_TO_PROCESS:
    print(f"Processing country: {country}")

    # Download wildlife crossings
    crossing_query = get_wildlife_crossing_query(
        country, timeout=240, bridges_only=BRIDGES_ONLY
    )
    response = r.get(OVERPASS_API_URL, params={"data": crossing_query})
    response.raise_for_status()
    data = response.json()

    features = []
    for el in data["elements"]:
        tags = el.get("tags", {})

        # Node → Point
        if el["type"] == "node":
            geometry = Point(el["lon"], el["lat"])
            features.append({"id": el["id"], "geometry": geometry, **tags})

        # Way → LineString or Polygon
        elif el["type"] == "way" and "geometry" in el:
            coords = [(pt["lon"], pt["lat"]) for pt in el["geometry"]]
            if len(coords) >= 3 and coords[0] == coords[-1]:
                geometry = Polygon(coords)  # Closed → Polygon
            else:
                geometry = LineString(coords)  # Open → LineString
            features.append({"id": el["id"], "geometry": geometry, **tags})

        # Relation → MultiPolygon (if members have geometry)
        elif el["type"] == "relation" and "members" in el:
            polygons = []
            for member in el["members"]:
                if member["type"] == "way" and "geometry" in member:
                    coords = [(pt["lon"], pt["lat"]) for pt in member["geometry"]]
                    if len(coords) >= 3:
                        polygons.append(Polygon(coords))
            if polygons:
                geometry = MultiPolygon(polygons) if len(polygons) > 1 else polygons[0]
                features.append({"id": el["id"], "geometry": geometry, **tags})

    # --- Create GeoDataFrame ---
    if features:
        print(f"Found {len(features)} wildlife crossings.")
        gdf = gpd.GeoDataFrame(features, geometry="geometry", crs="EPSG:4326")
    else:
        print("No wildlife crossings found.")

    # --- Reproject to EPSG:3035 (ETRS89 / LAEA Europe) ---
    if not gdf.empty:
        gdf = gdf.to_crs(epsg=CRS.split(":")[1])
        print(f"Reprojected to CRS: {CRS}")

    os.makedirs(GEODATA_OUTPUT_DIR, exist_ok=True)

    # Save the GeoDataFrame to a file
    output_file = os.path.join(
        GEODATA_OUTPUT_DIR,
        f"{country.lower()}_wildlife_crossings_osm{bridges_string}.gpkg",
    )
    gdf.to_file(output_file, driver="GPKG")
    print(f"Saved GeoDataFrame to {output_file}")

Processing country: CH


INFO:pyogrio._io:Created 18 records


Found 18 wildlife crossings.
Reprojected to CRS: EPSG:3035
Saved GeoDataFrame to ../data/geodata/ch_wildlife_crossings_osm_bridges_only.gpkg


### Download WMS Images

#### WMS Connection

In [5]:
wms = WebMapService(
    WMS_URL,
    version="1.3.0",
)
layer_name = list(wms.contents)[0]
print(layer_name)

VHR_2021_LAEA


#### Execution

In [17]:
for f in SOURCE_GPKGS:
    print(f"Processing source geometry file: {os.path.basename(f)}")
    gdf = gpd.read_file(f)
    gdf = gdf[
        (gdf.geom_type == "Polygon")
        | (gdf.geom_type == "Point") & (gdf["bridge"] == "yes")
    ]
    print(f"Number of geometries to process: {len(gdf)}")

    gdf_centroids = gdf.to_crs(epsg=3035).centroid
    areas_of_interest = get_crop_bboxes(gdf_centroids, BUFFER_SIZE)

    output_subdir = os.path.join(
        IMAGE_OUTPUT_DIR,
        os.path.splitext(os.path.basename(f))[0] + "_" + str(BUFFER_SIZE),
    )
    os.makedirs(output_subdir, exist_ok=True)
    os.makedirs(os.path.join(output_subdir, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_subdir, "labels"), exist_ok=True)

    # for every point create a square around it and request a WMS crop centered on the point
    img_format = "image/jpeg"
    img_size = (
        int(BUFFER_SIZE * 2 / PIXEL_RESOLUTION_M),
        int(BUFFER_SIZE * 2 / PIXEL_RESOLUTION_M),
    )  # width, height in pixels

    for idx, bbox in enumerate(areas_of_interest):
        # Request WMS image
        img = wms.getmap(
            layers=[layer_name],
            srs="EPSG:3035",
            bbox=bbox,
            size=img_size,
            format=img_format,
            transparent=True,
        )

        # Save image to file
        img_data = img.read()
        img_filename = os.path.join(output_subdir, "images", f"crossing_{idx}.png")
        with open(img_filename, "wb") as f:
            f.write(img_data)

        # save polygon label to file in yolo format
        label_filename = os.path.join(output_subdir, "labels", f"crossing_{idx}.txt")
        polygon = gdf.geometry.iloc[idx]

        # Convert polygon to YOLO segmentation format
        yolo_line = polygon_to_yolo_segmentation(polygon, bbox, class_id=0)

        if yolo_line:
            with open(label_filename, "w") as f:
                f.write(yolo_line + "\n")
            print(f"Saved WMS crop and label for crossing {idx} to {img_filename}")
        else:
            print(f"Warning: Could not convert geometry {idx} to YOLO format")

Processing source geometry file: ch_wildlife_crossings_osm_bridges_only.gpkg
Number of geometries to process: 12
Saved WMS crop and label for crossing 0 to ../data/satellite_images/ch_wildlife_crossings_osm_bridges_only_640\images\crossing_0.png
Saved WMS crop and label for crossing 1 to ../data/satellite_images/ch_wildlife_crossings_osm_bridges_only_640\images\crossing_1.png
Saved WMS crop and label for crossing 2 to ../data/satellite_images/ch_wildlife_crossings_osm_bridges_only_640\images\crossing_2.png
Saved WMS crop and label for crossing 3 to ../data/satellite_images/ch_wildlife_crossings_osm_bridges_only_640\images\crossing_3.png
Saved WMS crop and label for crossing 4 to ../data/satellite_images/ch_wildlife_crossings_osm_bridges_only_640\images\crossing_4.png
Saved WMS crop and label for crossing 5 to ../data/satellite_images/ch_wildlife_crossings_osm_bridges_only_640\images\crossing_5.png
Saved WMS crop and label for crossing 6 to ../data/satellite_images/ch_wildlife_crossings

### Road

#### Export LineStrings from OSM

In [7]:
for country in COUNTRIES_TO_PROCESS:
    print(f"Processing country for streets: {country}")

    # Download street data
    street_query = get_street_query(
        extent_country=country,
        street_type_depth=STREET_TYPE_DEPTH,
        timeout=OVERPASS_TIMEOUT,
    )
    response = r.get(OVERPASS_API_URL, params={"data": street_query})
    response.raise_for_status()
    data = response.json()

    features = []
    for el in data["elements"]:
        if el["type"] == "way" and "geometry" in el:
            coords = [(pt["lon"], pt["lat"]) for pt in el["geometry"]]
            # Only open ways → LineString
            if len(coords) >= 2 and coords[0] != coords[-1]:
                features.append(
                    {
                        "id": el["id"],
                        "geometry": LineString(coords),
                        **el.get("tags", {}),
                    }
                )

    # --- Build GeoDataFrame ---
    if not features:
        print("No LineString features found!")
        gdf = gpd.GeoDataFrame(columns=["id", "geometry"])
    else:
        gdf = gpd.GeoDataFrame(features, geometry="geometry", crs="EPSG:4326")
        print(f"Found {len(gdf)} LineString features.")

    # --- Prepare output folder ---
    output_dir = os.path.abspath(os.path.join(GEODATA_OUTPUT_DIR))
    os.makedirs(output_dir, exist_ok=True)

    # --- Reproject to EPSG:3035 (ETRS89 / LAEA Europe) ---
    if not gdf.empty:
        gdf = gdf.to_crs(epsg=CRS.split(":")[1])
        print(f"Reprojected to CRS:{CRS}.")

    # --- Save GPKG ---
    output_file = os.path.join(
        output_dir, f"{country}_highway_depth_{STREET_TYPE_DEPTH}.gpkg"
    )
    gdf.to_file(output_file, driver="GPKG")
    print(f"Data saved to {output_file}")

Processing country for streets: CH
Found 14716 LineString features.
Reprojected to CRS:EPSG:3035.


INFO:pyogrio._io:Created 14,716 records


Data saved to c:\code\cassda-zertifikatsarbeit\data\geodata\CH_highway_depth_1.gpkg
